<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/hellaswag_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
import glob

original_file = "/content/hellaswag_rate_0.1.jsonl"
esl_files = sorted(glob.glob("/content/*_A.jsonl"))

print("Original file exists:", os.path.exists(original_file))
print("Number of ESL files:", len(esl_files))

for file in esl_files:
    print(os.path.basename(file))

Original file exists: True
Number of ESL files: 8
arabic_A.jsonl
chinese_A.jsonl
french_A.jsonl
german_A.jsonl
japanese_A.jsonl
portuguese_A.jsonl
russian_A.jsonl
spanish_A.jsonl


In [12]:
import json
import pandas as pd

original_records = []

with open(original_file, "r", encoding="utf-8-sig") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)

        endings = record.get("endings", [])
        label = int(record["label"])

        original_records.append({
            "split": record.get("split"),
            "ind": record.get("ind"),
            "source_id": record.get("source_id"),
            "activity_label": record.get("activity_label"),
            "original_prompt": record.get("original_text_backup"),
            "endings": endings,
            "label": label,
            "gold_answer": endings[label]
        })

original_df = pd.DataFrame(original_records)

print("Original records:", len(original_df))
display(original_df.head())

Original records: 39905


,split,ind,source_id,activity_label,original_prompt,endings,label,gold_answer
0,train,4,activitynet~v_-1IBHYS3L-Y,Removing ice from car,"Then, the man writes over the snow covering th...","[, the man adds wax to the windshield and cuts...",3,", the man continues removing the snow on his car."
1,train,8,activitynet~v_-2dxp-mv2zo,Baking cookies,A female chef in white uniform shows a stack o...,"[contain egg yolks and baking soda., are then ...",3,are filled with pastries and loaded into the o...
2,train,9,activitynet~v_-2dxp-mv2zo,Baking cookies,A female chef in white uniform shows a stack o...,[is seen moving on a board and cutting out its...,3,is used to cut cylinder shaped dough into rounds.
3,train,12,activitynet~v_-2dxp-mv2zo,Baking cookies,A tray of potatoes is loaded into the oven and...,"[is placed onto a baked potato., , ls, and pic...",3,is prepared then it is removed from the oven b...
4,train,27,activitynet~v_-JqLjPz-07E,Getting a haircut,The man in the center is demonstrating a hairs...,[is standing on the sponge cutting the hair of...,2,sits on the chair next to the sink.


In [13]:
original_df.to_csv(
    "/content/hellaswag_original_reference.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", len(original_df), "original records")

Saved: 39905 original records


In [14]:
import json
import os
import pandas as pd

esl_records = []

for file_path in esl_files:

    # arabic_A.jsonl -> arabic
    language = (
        os.path.basename(file_path)
        .replace("_A.jsonl", "")
    )

    with open(file_path, "r", encoding="utf-8-sig") as f:

        for line in f:

            if not line.strip():
                continue

            record = json.loads(line)

            endings = record.get("endings", [])
            label = int(record["label"])

            esl_records.append({
                "esl_language": language,
                "split": record.get("split"),
                "ind": record.get("ind"),
                "source_id": record.get("source_id"),
                "activity_label": record.get("activity_label"),
                "modified_prompt": record.get("ctx"),
                "endings": endings,
                "label": label,
                "gold_answer": endings[label]
            })

esl_df = pd.DataFrame(esl_records)

print("Total ESL records:", len(esl_df))
print("\nRecords by language:")
print(esl_df.groupby("esl_language").size())

display(esl_df.head())

Total ESL records: 60744

Records by language:
esl_language
arabic        7593
chinese       7593
french        7593
german        7593
japanese      7593
portuguese    7593
russian       7593
spanish       7593
dtype: int64


,esl_language,split,ind,source_id,activity_label,modified_prompt,endings,label,gold_answer
0,arabic,val,24,activitynet~v_-JhWjGDPHMY,Roof shingle removal,on a roof is sitting the man.,"[is using wrap to wrap a pair of skis., is rip...",3,starts pulling up roofing on a roof.
1,arabic,val,106,activitynet~v_-xQvJmC2jhk,Canoeing,Two women of a child are shown a canoe while a...,[are then shown paddling down a river in a boa...,2,sit in a canoe while the man paddles.
2,arabic,val,114,activitynet~v_-zHX3Gdx6I4,High jump,"""An boys am running down a track.""","[runs into a car., gets in a mat., lifts his b...",2,lifts his body above the height of a pole.
3,arabic,val,116,activitynet~v_-zHX3Gdx6I4,High jump,"""The boy lifts his body above the height a pol...","[turns his body around on the mat., gets up fr...",1,gets up from the mat.
4,arabic,val,116,activitynet~v_-zHX3Gdx6I4,High jump,The boy did land on his back a mats plus red t...,"[turns his body around on the mat., gets up fr...",1,gets up from the mat.


In [15]:
match_keys = ["split", "ind", "source_id"]

original_duplicates = original_df.duplicated(
    subset=match_keys,
    keep=False
).sum()

esl_duplicates = esl_df.duplicated(
    subset=["esl_language"] + match_keys,
    keep=False
).sum()

print("Original duplicate rows:", original_duplicates)
print("ESL duplicate rows:", esl_duplicates)

Original duplicate rows: 0
ESL duplicate rows: 8382


In [16]:
duplicate_rows = esl_df[
    esl_df.duplicated(
        subset=["esl_language"] + match_keys,
        keep=False
    )
]

print(
    duplicate_rows.groupby("esl_language").size()
)

display(
    duplicate_rows[
        [
            "esl_language",
            "split",
            "ind",
            "source_id",
            "modified_prompt"
        ]
    ].head(20)
)

esl_language
arabic        1362
chinese        836
french         494
german        1106
japanese      1158
portuguese    1184
russian        948
spanish       1294
dtype: int64


,esl_language,split,ind,source_id,modified_prompt
3,arabic,val,116,activitynet~v_-zHX3Gdx6I4,"""The boy lifts his body above the height a pol..."
4,arabic,val,116,activitynet~v_-zHX3Gdx6I4,The boy did land on his back a mats plus red t...
10,arabic,val,186,activitynet~v_0bosp4-pyTM,"The stone particles stick the piece of wood, p..."
11,arabic,val,186,activitynet~v_0bosp4-pyTM,"""Did take a knife and did sharpen it against a..."
18,arabic,val,378,activitynet~v_2rA5pyel_NE,Children bring out desert for a famil member.
20,arabic,val,385,activitynet~v_33SI8z8PovA,"A female, black, is shown a room black around ..."
22,arabic,val,378,activitynet~v_2rA5pyel_NE,"""The young boy and girl are standing an sink w..."
24,arabic,val,385,activitynet~v_33SI8z8PovA,"""A mother instructs them on how to brush their..."
25,arabic,val,402,activitynet~v_3HBAcaU552I,get these one some water to gargle their mouth...
27,arabic,val,479,activitynet~v_4Xvn1xXvYdU,"""Speaking archery, man brown jacket pulled slo..."


In [17]:
duplicate_keys = ["esl_language", "split", "ind", "source_id"]

dup_df = esl_df[
    esl_df.duplicated(
        subset=duplicate_keys,
        keep=False
    )
].copy()

dup_summary = (
    dup_df.groupby(duplicate_keys)
    .agg(
        rows=("modified_prompt", "size"),
        unique_prompts=("modified_prompt", "nunique")
    )
    .reset_index()
)

print("Duplicate groups:", len(dup_summary))
print(
    "Same-prompt groups:",
    (dup_summary["unique_prompts"] == 1).sum()
)
print(
    "Different-prompt groups:",
    (dup_summary["unique_prompts"] > 1).sum()
)

display(dup_summary.head(20))

Duplicate groups: 4191
Same-prompt groups: 0
Different-prompt groups: 4191


,esl_language,split,ind,source_id,rows,unique_prompts
0,arabic,val,116,activitynet~v_-zHX3Gdx6I4,2,2
1,arabic,val,124,wikihow~206,2,2
2,arabic,val,132,wikihow~218,2,2
3,arabic,val,186,activitynet~v_0bosp4-pyTM,2,2
4,arabic,val,305,wikihow~497,2,2
5,arabic,val,359,wikihow~583,2,2
6,arabic,val,361,wikihow~586,2,2
7,arabic,val,378,activitynet~v_2rA5pyel_NE,2,2
8,arabic,val,381,wikihow~615,2,2
9,arabic,val,385,activitynet~v_33SI8z8PovA,2,2


In [18]:
duplicate_keys = [
    "esl_language",
    "split",
    "ind",
    "source_id"
]

duplicate_mask = esl_df.duplicated(
    subset=duplicate_keys,
    keep=False
)

# 保留没有重复版本的记录
esl_clean_df = esl_df[
    ~duplicate_mask
].copy()

print("Before cleaning:", len(esl_df))
print("After cleaning:", len(esl_clean_df))

print("\nClean records by language:")
print(
    esl_clean_df.groupby("esl_language").size()
)

Before cleaning: 60744
After cleaning: 52362

Clean records by language:
esl_language
arabic        6231
chinese       6757
french        7099
german        6487
japanese      6435
portuguese    6409
russian       6645
spanish       6299
dtype: int64


In [19]:
match_keys = ["split", "ind", "source_id"]

matched_df = esl_clean_df.merge(
    original_df[match_keys + ["original_prompt"]],
    on=match_keys,
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Total ESL rows:", len(matched_df))
print("\nMatch result:")
print(matched_df["_merge"].value_counts())

print("\nUnmatched rows by language:")
print(
    matched_df[matched_df["_merge"] == "left_only"]
    .groupby("esl_language")
    .size()
)

display(matched_df.head())

Total ESL rows: 52362

Match result:
_merge
left_only     52362
right_only        0
both              0
Name: count, dtype: int64

Unmatched rows by language:
esl_language
arabic        6231
chinese       6757
french        7099
german        6487
japanese      6435
portuguese    6409
russian       6645
spanish       6299
dtype: int64


,esl_language,split,ind,source_id,activity_label,modified_prompt,endings,label,gold_answer,original_prompt,_merge
0,arabic,val,24,activitynet~v_-JhWjGDPHMY,Roof shingle removal,on a roof is sitting the man.,"[is using wrap to wrap a pair of skis., is rip...",3,starts pulling up roofing on a roof.,NaN,left_only
1,arabic,val,106,activitynet~v_-xQvJmC2jhk,Canoeing,Two women of a child are shown a canoe while a...,[are then shown paddling down a river in a boa...,2,sit in a canoe while the man paddles.,NaN,left_only
2,arabic,val,114,activitynet~v_-zHX3Gdx6I4,High jump,"""An boys am running down a track.""","[runs into a car., gets in a mat., lifts his b...",2,lifts his body above the height of a pole.,NaN,left_only
3,arabic,val,149,activitynet~v_0RUMAGGab1k,Playing harmonica,"""An man is standing in front a camera plus he ...",[begins to play the harmonica with his body wh...,2,rocks back and forth to the music as he goes.,NaN,left_only
4,arabic,val,170,activitynet~v_0WVkoTBmhA0,Sumo,A cartoon film person walking around and a roc...,[fight robots of evil and ends with a to be co...,0,fight robots of evil and ends with a to be con...,NaN,left_only


In [20]:
print("Original split:")
print(original_df["split"].value_counts())

print("\nESL split:")
print(esl_clean_df["split"].value_counts())

Original split:
split
train    39905
Name: count, dtype: int64

ESL split:
split
val    52362
Name: count, dtype: int64


In [21]:
import os

val_file = "/content/hellaswag_val.jsonl"

print("File exists:", os.path.exists(val_file))

File exists: True


In [22]:
import json
from collections import Counter

splits = []

with open(val_file, "r", encoding="utf-8-sig") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            splits.append(record.get("split"))

print(Counter(splits))

Counter({'val': 10042})


In [23]:
import json
import pandas as pd

val_records = []

with open(val_file, "r", encoding="utf-8-sig") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        endings = record.get("endings", [])
        label = int(record["label"])

        val_records.append({
            "split": record.get("split"),
            "ind": record.get("ind"),
            "source_id": record.get("source_id"),
            "activity_label": record.get("activity_label"),
            "original_prompt": record.get("ctx"),
            "endings": endings,
            "label": label,
            "gold_answer": endings[label]
        })

# 覆盖之前的 train 版 original_df
original_df = pd.DataFrame(val_records)

print("Original records:", len(original_df))
print(original_df["split"].value_counts())

display(original_df.head())

Original records: 10042
split
val    10042
Name: count, dtype: int64


,split,ind,source_id,activity_label,original_prompt,endings,label,gold_answer
0,val,24,activitynet~v_-JhWjGDPHMY,Roof shingle removal,A man is sitting on a roof. he,"[is using wrap to wrap a pair of skis., is rip...",3,starts pulling up roofing on a roof.
1,val,92,activitynet~v_-lJS58hyo1c,Clean and jerk,A lady walks to a barbell. She bends down and ...,"[swings and lands in her arms., pulls the barb...",3,stands and lifts the weight over her head.
2,val,106,activitynet~v_-xQvJmC2jhk,Canoeing,Two women in a child are shown in a canoe whil...,[are then shown paddling down a river in a boa...,2,sit in a canoe while the man paddles.
3,val,114,activitynet~v_-zHX3Gdx6I4,High jump,A boy is running down a track. the boy,"[runs into a car., gets in a mat., lifts his b...",2,lifts his body above the height of a pole.
4,val,116,activitynet~v_-zHX3Gdx6I4,High jump,The boy lifts his body above the height of a p...,"[turns his body around on the mat., gets up fr...",1,gets up from the mat.


In [24]:
original_df.to_csv(
    "/content/hellaswag_val_original_reference.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully.")

Saved successfully.


In [25]:
match_keys = ["split", "ind", "source_id"]

matched_df = esl_clean_df.merge(
    original_df[
        match_keys + ["original_prompt", "label", "endings"]
    ].rename(columns={
        "label": "original_label",
        "endings": "original_endings"
    }),
    on=match_keys,
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Total rows:", len(matched_df))
print("\nMatch result:")
print(matched_df["_merge"].value_counts())

Total rows: 52362

Match result:
_merge
both          52362
left_only         0
right_only        0
Name: count, dtype: int64


In [26]:
matched_df["label_match"] = (
    matched_df["label"] == matched_df["original_label"]
)

matched_df["endings_match"] = matched_df.apply(
    lambda row: row["endings"] == row["original_endings"],
    axis=1
)

print("Label check:")
print(matched_df["label_match"].value_counts())

print("\nEndings check:")
print(matched_df["endings_match"].value_counts())

Label check:
label_match
True    52362
Name: count, dtype: int64

Endings check:
endings_match
True    52362
Name: count, dtype: int64


In [27]:
key_cols = ["split", "ind", "source_id"]

common_items = (
    matched_df
    .groupby(key_cols)["esl_language"]
    .nunique()
    .reset_index(name="language_count")
)

common_items = common_items[
    common_items["language_count"] == 8
].copy()

print("Original items shared by all 8 languages:", len(common_items))

display(common_items.head())

Original items shared by all 8 languages: 2680


,split,ind,source_id,language_count
0,val,1,wikihow~4,8
3,val,24,activitynet~v_-JhWjGDPHMY,8
7,val,76,wikihow~118,8
8,val,87,wikihow~143,8
16,val,147,wikihow~238,8


In [28]:
# 固定随机抽 10 个共同 original
sampled_keys = (
    common_items
    .sample(n=10, random_state=42)
    [key_cols]
    .reset_index(drop=True)
)

# 给 10 个 original 编号
sampled_keys["original_sample_id"] = [
    f"HS_ORIG_{i:02d}" for i in range(1, 11)
]

# 取出每个 original 的 8 种语言版本
sampled_df = matched_df.merge(
    sampled_keys,
    on=key_cols,
    how="inner"
)

sampled_df = sampled_df.sort_values(
    ["original_sample_id", "esl_language"]
).reset_index(drop=True)

print("Unique originals:", sampled_df["original_sample_id"].nunique())
print("Total rows:", len(sampled_df))

print("\nRows by language:")
print(sampled_df.groupby("esl_language").size())

print("\nRows per original:")
print(sampled_df.groupby("original_sample_id").size())

display(
    sampled_df[
        [
            "original_sample_id",
            "esl_language",
            "ind",
            "original_prompt",
            "modified_prompt"
        ]
    ].head(16)
)

Unique originals: 10
Total rows: 80

Rows by language:
esl_language
arabic        10
chinese       10
french        10
german        10
japanese      10
portuguese    10
russian       10
spanish       10
dtype: int64

Rows per original:
original_sample_id
HS_ORIG_01    8
HS_ORIG_02    8
HS_ORIG_03    8
HS_ORIG_04    8
HS_ORIG_05    8
HS_ORIG_06    8
HS_ORIG_07    8
HS_ORIG_08    8
HS_ORIG_09    8
HS_ORIG_10    8
dtype: int64


,original_sample_id,esl_language,ind,original_prompt,modified_prompt
0,HS_ORIG_01,arabic,23671,[header] How to deal with oily straight hair [...,How to deal greasy straight hair. Eat food ric...
1,HS_ORIG_01,chinese,23671,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...
2,HS_ORIG_01,french,23671,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...
3,HS_ORIG_01,german,23671,[header] How to deal with oily straight hair [...,How to deal with grease straight hair. Eat foo...
4,HS_ORIG_01,japanese,23671,[header] How to deal with oily straight hair [...,How to deal with greasy straight haire? Eat fo...
5,HS_ORIG_01,portuguese,23671,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...
6,HS_ORIG_01,russian,23671,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...
7,HS_ORIG_01,spanish,23671,[header] How to deal with oily straight hair [...,How to deal with greasy straight a hair. Eat f...
8,HS_ORIG_02,arabic,35533,[header] How to make apple jam [title] Peel an...,"""You removing skin and cutting your apple, plu..."
9,HS_ORIG_02,chinese,35533,[header] How to make apple jam [title] Peel an...,"You removing skin and cutting your apples, plu..."


In [29]:
output_file = "/content/hellaswag_10_originals_x_8_languages.csv"

sampled_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_file)
print("Total rows:", len(sampled_df))

Saved: /content/hellaswag_10_originals_x_8_languages.csv
Total rows: 80


In [30]:
import os
import ast
import pandas as pd
from google.colab import files


# ============================================================
# 1. Load the sampled HellaSwag data
# ============================================================


if "sampled_df" in globals():
    review_df = sampled_df.copy()


else:
    input_file = (
        "/content/"
        "hellaswag_10_originals_x_8_languages.csv"
    )

    review_df = pd.read_csv(input_file)


# ============================================================
# 2. Sort the 10 originals and 8 language versions
# ============================================================

review_df = review_df.sort_values(
    [
        "original_sample_id",
        "esl_language"
    ]
).reset_index(drop=True)


# 给每一行添加独立编号
review_df.insert(
    0,
    "review_row_id",
    [
        f"HS_REVIEW_{i:03d}"
        for i in range(1, len(review_df) + 1)
    ]
)


# ============================================================
# 3. Split the four HellaSwag answer choices
# ============================================================

def parse_endings(value):

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(str(value))

        if isinstance(parsed, list):
            return parsed

    except Exception:
        pass

    return []


review_df["_parsed_endings"] = review_df[
    "endings"
].apply(parse_endings)


for i in range(4):

    review_df[f"ending_{i + 1}"] = review_df[
        "_parsed_endings"
    ].apply(
        lambda choices: (
            choices[i]
            if len(choices) > i
            else ""
        )
    )


review_df = review_df.drop(
    columns=["_parsed_endings"]
)



review_df["correct_option"] = (
    pd.to_numeric(
        review_df["label"],
        errors="coerce"
    )
    + 1
)


# ============================================================
# 4. Add human-review columns
# ============================================================

review_columns = [
    "R1_Meaning",
    "R2_Meaning",
    "Final_Meaning",

    "R1_Key_Info",
    "R2_Key_Info",
    "Final_Key_Info",

    "R1_Answer_Preserved",
    "R2_Answer_Preserved",
    "Final_Answer_Preserved",

    "R1_Realism",
    "R2_Realism",
    "Final_Realism",

    "R1_Readability",
    "R2_Readability",
    "Final_Readability",

    "R1_Comments",
    "R2_Comments",
    "Final_Comments"
]


for column in review_columns:

    if column not in review_df.columns:
        review_df[column] = ""


# ============================================================
# 5. Keep and arrange the useful columns
# ============================================================

final_columns = [
    "review_row_id",
    "original_sample_id",
    "esl_language",
    "split",
    "ind",
    "source_id",
    "activity_label",

    "original_prompt",
    "modified_prompt",

    "ending_1",
    "ending_2",
    "ending_3",
    "ending_4",

    "correct_option",
    "gold_answer",

    "R1_Meaning",
    "R2_Meaning",
    "Final_Meaning",

    "R1_Key_Info",
    "R2_Key_Info",
    "Final_Key_Info",

    "R1_Answer_Preserved",
    "R2_Answer_Preserved",
    "Final_Answer_Preserved",

    "R1_Realism",
    "R2_Realism",
    "Final_Realism",

    "R1_Readability",
    "R2_Readability",
    "Final_Readability",

    "R1_Comments",
    "R2_Comments",
    "Final_Comments"
]


# 只保留实际存在的列
final_columns = [
    column
    for column in final_columns
    if column in review_df.columns
]

review_df = review_df[
    final_columns
].copy()


# ============================================================
# 6. Final checks
# ============================================================

print("Total review rows:", len(review_df))

print("\nRows by language:")
print(
    review_df.groupby(
        "esl_language"
    ).size()
)

print("\nRows per original:")
print(
    review_df.groupby(
        "original_sample_id"
    ).size()
)


assert len(review_df) == 80

assert (
    review_df.groupby("esl_language").size()
    == 10
).all()

assert (
    review_df.groupby("original_sample_id").size()
    == 8
).all()


# ============================================================
# 7. Save the Google Sheets review CSV
# ============================================================

output_file = (
    "/content/"
    "hellaswag_10_originals_x_8_languages_review.csv"
)

review_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print("\n✅ Review CSV saved:")
print(output_file)


# Preview
display(review_df.head(8))


# ============================================================
# 8. Download the CSV
# ============================================================

files.download(output_file)

Total review rows: 80

Rows by language:
esl_language
arabic        10
chinese       10
french        10
german        10
japanese      10
portuguese    10
russian       10
spanish       10
dtype: int64

Rows per original:
original_sample_id
HS_ORIG_01    8
HS_ORIG_02    8
HS_ORIG_03    8
HS_ORIG_04    8
HS_ORIG_05    8
HS_ORIG_06    8
HS_ORIG_07    8
HS_ORIG_08    8
HS_ORIG_09    8
HS_ORIG_10    8
dtype: int64

✅ Review CSV saved:
/content/hellaswag_10_originals_x_8_languages_review.csv


,review_row_id,original_sample_id,esl_language,split,ind,source_id,activity_label,original_prompt,modified_prompt,ending_1,...,Final_Answer_Preserved,R1_Realism,R2_Realism,Final_Realism,R1_Readability,R2_Readability,Final_Readability,R1_Comments,R2_Comments,Final_Comments
0,HS_REVIEW_001,HS_ORIG_01,arabic,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal greasy straight hair. Eat food ric...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,
1,HS_REVIEW_002,HS_ORIG_01,chinese,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,
2,HS_REVIEW_003,HS_ORIG_01,french,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,
3,HS_REVIEW_004,HS_ORIG_01,german,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal with grease straight hair. Eat foo...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,
4,HS_REVIEW_005,HS_ORIG_01,japanese,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal with greasy straight haire? Eat fo...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,
5,HS_REVIEW_006,HS_ORIG_01,portuguese,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,
6,HS_REVIEW_007,HS_ORIG_01,russian,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal with greasy straight hair. Eat foo...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,
7,HS_REVIEW_008,HS_ORIG_01,spanish,val,23671,wikihow~39096,Personal Care and Style,[header] How to deal with oily straight hair [...,How to deal with greasy straight a hair. Eat f...,[title] Cut out unhealthy food. [step] Avoid j...,...,,,,,,,,,,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>